In [10]:
# Cell 1 — Import Required Libraries and Configure LLM Access
import ollama

response = ollama.chat(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": "Why is the sky blue?"
        }
    ]
)

print(response["message"]["content"])

The sky appears blue to us because of a phenomenon called scattering. When sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen and oxygen. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described the phenomenon in the late 19th century.

Here's a simplified explanation of the process:

1. **Sunlight Enters the Atmosphere**: When sunlight enters the Earth's atmosphere, it is made up of a range of colors, including red, orange, yellow, green, blue, indigo, and violet.

2. **Scattering Occurs**: The light is then scattered by the tiny molecules of gases in the atmosphere. This scattering affects the shorter wavelengths more than the longer wavelengths.

3. **Blue Light Dominates**: Because blue light has a shorter wavelength, it is scattered more than the other colors, m

In [11]:
# Cell 2 — Load and Validate Cleaned Data
import pandas as pd
df = pd.read_csv("../data/processed/customer_support_tickets_clean.csv")

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = df["Ticket Type"].fillna("").str.strip().eq("").sum()
print(f"Invalid or empty Ticket Type values: {invalid_type}")

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Description values: {invalid_description}")

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [12]:
# Cell 3 — Define LLM Ticket Analysis Requirements

ticket_categories = [
    "Technical Issue",
    "Refund Request",
    "Cancellation Request",
    "Billing Inquiry",
    "Product Inquiry"
]

# Required LLM outputs:
# Summary   = short description of the customer's problem
# Category  = one of the five ticket_categories above
# Sentiment = customer's expressed tone/sentiment
# Urgency   = estimated urgency of the problem

In [13]:
# Cell 4 — Select Sample Ticket Descriptions

descriptions = df["Ticket Description"].sample(
    10,
    random_state=42
)

display(descriptions)

4830    I'm having an issue with the {product_purchase...
7075    I'm having trouble connecting my {product_purc...
4715    I'm having an issue with the {product_purchase...
2022    I'm having an issue with the {product_purchase...
676     I'm having an issue with the {product_purchase...
2283    I've encountered a data loss issue with my {pr...
5076    I'm having an issue with the {product_purchase...
2476    I'm having an issue with the {product_purchase...
6847    I'm encountering a software bug in the {produc...
511     I'm having an issue with the {product_purchase...
Name: Ticket Description, dtype: object

In [14]:
# Cell 5 — Create the Ticket Analysis Prompt

def create_prompt(ticket_description):
    prompt = f"""
Analyze the following customer-support ticket.

Ticket description:
{ticket_description}

Classify the ticket using exactly one of these categories:
- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

Provide the following information:

Summary: A short description of the customer's problem.
Category: One of the five allowed ticket categories.
Sentiment: The customer's expressed sentiment.
Urgency: The estimated urgency of the problem.
"""
    return prompt

In [15]:
# Cell 6 — Send One Ticket to the LLM

sample_ticket = descriptions.iloc[0]

ticket_prompt = create_prompt(sample_ticket)

ticket_response = ollama.chat(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": ticket_prompt
        }
    ]
)

In [16]:
# Cell 7 — Inspect the Raw LLM Response

raw_response = ticket_response["message"]["content"]

print("Original Ticket:")
print(sample_ticket)

print("\nLLM Response:")
print(raw_response)

Original Ticket:
I'm having an issue with the {product_purchased}. Please assist. I'm using xda-developer for something different. If there are issues with the {product_purchased} it's likely you are not using the I've tried clearing the cache and data for the {product_purchased} app, but the issue persists.

LLM Response:
Here's the analysis of the customer-support ticket:

**Summary**: The customer is experiencing an issue with the product they purchased and has tried troubleshooting by clearing the cache and data, but the problem persists.

**Category**: Technical Issue

**Sentiment**: Frustrated/Neutral (The customer seems frustrated with the issue, but also provides a clear and specific description of what they've tried to troubleshoot, indicating a neutral attitude towards the support team.)

**Urgency**: Medium (The customer has already tried troubleshooting, but the issue still persists, indicating that they need assistance to resolve the problem as soon as possible.)


In [17]:
# Cell 8 — Test the Prompt with Multiple Ticket Descriptions

test_descriptions = [
    "My account shows the payment for anti-virus system was made twice while I was attempting several times. When am I expected to receive my money back?",
    "I want to verify why my computer freezes while I run my browser and does not save the last edits in my reports.",
    "My browser does not log in after I submit the right passcode. Can you verify? Thanks.",
    "I would like to cancel the plan for this data cleaning service."
]

more_responses = []

for description in test_descriptions:
    current_prompt = create_prompt(description)

    response = ollama.chat(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": current_prompt
            }
        ]
    )

    more_responses.append(
        response["message"]["content"]
    )

In [ ]:
# Cell 9 — Inspect Summary, Category, Sentiment, and Urgency Results

for i, (description, response_text) in enumerate(
    zip(test_descriptions, more_responses),
    start=1
):
    print(f"===== Ticket {i} =====")

    print("\nTicket Description:")
    print(description)

    print("\nLLM Analysis:")
    print(response_text)

    print("\n")

===== Ticket 1 =====

Ticket Description:
My account shows the payment for anti-virus system was made twice while I was attempting several times. When am I expected to receive my money back?

LLM Analysis:
Here is the analysis of the customer-support ticket:

**Summary**: The customer's account shows a duplicate payment for an anti-virus system, and they want to know when they will receive their money back.

**Category**: Billing Inquiry

**Sentiment**: Neutral/Expectant (the customer is neutral about the situation, but expects a solution and a refund)

**Urgency**: Medium (the customer wants their money back, but the issue is not urgent, as they have already made the payment)


===== Ticket 2 =====

Ticket Description:
I want to verify why my computer freezes while I run my browser and does not save the last edits in my reports.

LLM Analysis:
Here is the analysis of the customer-support ticket:

**Summary:** The customer's computer freezes when running the browser and does not save t

: 

## Day 28 Conclusions

### LLM Ticket Analysis Pipeline

A local Llama 3.1 model was connected to the project using Ollama. A reusable prompt was created to analyze a customer-support ticket and produce four pieces of information:

- **Summary** — a short description of the customer's problem.
- **Category** — one of the five predefined Ticket Type categories.
- **Sentiment** — the customer's expressed sentiment.
- **Urgency** — the estimated urgency of the ticket.

The five allowed categories were:

- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

### Dataset Validation

The cleaned customer-support dataset contained:

- 0 duplicated tickets
- 0 invalid or empty Ticket Type values
- 0 invalid or empty Ticket Description values

This allowed the cleaned dataset to be reused for the LLM analysis.

### Initial LLM Test

The first sampled ticket described a persistent product problem after the customer had already attempted troubleshooting.

The LLM classified the ticket as **Technical Issue**, summarized the problem, described the sentiment as **Neutral**, and assigned **Medium** urgency.

This demonstrated that the prompt could produce all four requested outputs from a single ticket description.

### Multiple Ticket Tests

The prompt was then tested on four manually written ticket descriptions.

The LLM produced the following categories:

| Ticket | Main Issue | LLM Category | Urgency |
| --- | --- | --- | --- |
| 1 | Duplicate payment and refund question | Billing Inquiry | Medium |
| 2 | Computer freezing and unsaved work | Technical Issue | Medium |
| 3 | Unable to log in with correct passcode | Technical Issue | Medium |
| 4 | Request to cancel a service | Cancellation Request | Low |

For these examples, the selected categories were consistent with the main issue described in each ticket.

The LLM also generated a short summary, sentiment assessment, and urgency assessment for every tested ticket.

### Observations

The LLM consistently followed the requested four-part response format across the tested examples. However, the exact wording of the sentiment values was not standardized. For example, the model returned values such as **Annoyed/Frustrated**, **Frustrated**, **Neutral/Help-seeking**, and **Neutral**.

The responses are also currently returned as formatted text rather than fields that can be directly accessed by Python. Although the output is readable by a person, extracting individual values such as Category or Urgency programmatically would require additional processing.

The sampled dataset descriptions also contain placeholder text such as `{product_purchased}`, indicating that some source ticket descriptions are template-like. This should be considered when interpreting later evaluation results.

### Conclusion

Day 28 established a working LLM-based ticket analysis pipeline:

**Ticket Description → Prompt → Llama 3.1 → Summary + Category + Sentiment + Urgency**

The initial tests show that the LLM can perform multiple support-ticket analysis tasks from a single description and can restrict its category selection to the predefined Ticket Type categories in the examples tested.

However, Day 28 only tested a small number of examples and does not establish the overall accuracy of the LLM. The current free-text responses also need to be converted into a consistent structured format before they can be reliably processed and evaluated automatically.

The next step is to create structured LLM output so that Summary, Category, Sentiment, and Urgency can be accessed individually and used in later ML-versus-LLM evaluation.